In [1]:
# IMPORTS FROM THE OGDF LIBRARY
from ogdf_python import *
cppinclude("ogdf/energybased/FMMMLayout.h")
cppinclude("ogdf/layered/SugiyamaLayout.h")
cppinclude("ogdf/layered/MedianHeuristic.h")
cppinclude("ogdf/layered/OptimalHierarchyLayout.h")
cppinclude("ogdf/layered/OptimalRanking.h")
cppinclude("ogdf/fileformats/GraphIO.h")
cppinclude("ogdf/orthogonal/OrthoLayout.h")
cppinclude("ogdf/planarity/PlanarizationLayout.h")
cppinclude("ogdf/planarity/SubgraphPlanarizer.h")

import networkx as nx
import cairosvg
import os

def disown(obj):
    obj.__python_owns__ = False
    return obj

(Re-)building pre-compiled headers (options: -O2 -march=native); this may take a minute ...


c:\Users\FabrizioMontecchiani\AppData\Local\Programs\Python\Python312\Lib\site-packages\cppyy_backend\loader.py:138: UserWarning: No precompiled header available (failed to build); this may impact performance.
  warnings.warn('No precompiled header available (%s); this may impact performance.' % msg)


RuntimeError: could not load cppyy_backend library, details:
  Could not find module 'libcppyy_backend.dll' (or one of its dependencies). Try using the full path with constructor syntax.
  Could not find module 'c:\Users\FabrizioMontecchiani\AppData\Local\Programs\Python\Python312\Lib\site-packages\cppyy_backend\lib\libcppyy_backend.cp312-win_amd64.pyd' (or one of its dependencies). Try using the full path with constructor syntax.
  Could not find module 'libcppyy_backend.cp312-win_amd64.pyd' (or one of its dependencies). Try using the full path with constructor syntax.
  [WinError 1114] Routine di inizializzazione della libreria di collegamento dinamico (DLL) non riuscita

In [26]:
# GLOBAL CONSTANTS FOR GRAPHIC FEATURES

FILL_COLOR = "#FFFF00"
WIDTH = 15.0
HEIGHT = 15.0
SHAPE = ogdf.Shape.Ellipse
BG = "#FFFFFF"
DPI = 32
OUTPUT_WIDTH = 1024

In [27]:
# FUNCTION TO IMPORT .LST FILES
def readFromAdjacencyList(filepath, G):
    file = open(filepath,'r')
    map = {}
    for row in file:
        parts = row.strip().split(":")
        if len(parts) < 2:
            break
        v = int(parts[0].strip())
        if v not in map.keys():
            n = G.newNode(v)
            map[v] = n
        neighbors = parts[1].strip().split(" ")
        for item in neighbors:
            u = int(item.strip())
            if u not in map.keys():
                n = G.newNode(u)
                map[u] = n
            if G.searchEdge(map[v],map[u],directed=False)==None:
                G.newEdge(map[v],map[u])
    file.close()

In [28]:
def readFromGML(filepath, G):
    H = nx.read_gml(filepath)
    map = {}
    for v in H.nodes:
        n = G.newNode(int(v))
        map[v] = n
    for e in H.edges:
        G.newEdge(map[e[0]], map[e[1]])

In [29]:
# LAYERED LAYOUT FOR DIRECTED GRAPHS
def computeSugiyamaLayout(GA):
    SL = ogdf.SugiyamaLayout()
    SL.setRanking(disown(ogdf.OptimalRanking()))
    SL.setCrossMin(disown(ogdf.MedianHeuristic()))
    ohl = disown(ogdf.OptimalHierarchyLayout())
    ohl.layerDistance(30.0)
    ohl.nodeDistance(40.0)
    ohl.weightBalancing(0.8)
    SL.setLayout(ohl)
    SL.call(GA)

In [30]:
# FORCE-DIRECTED LAYOUT FOR UNDIRECTED GRAPHS
def computeFMMMLayout(GA):
    FMMM = ogdf.FMMMLayout()
    FMMM.useHighLevelOptions(True)
    FMMM.unitEdgeLength(30.0)
    FMMM.newInitialPlacement(True)
    FMMM.qualityVersusSpeed(ogdf.FMMMOptions.QualityVsSpeed.GorgeousAndEfficient)
    FMMM.call(GA)

In [31]:
# ORTHOGONAL LAYOUT FOR UNDIRECTED GRAPHS
def computeOrthoLayout(GA):
    pl = ogdf.PlanarizationLayout()
    ol = ogdf.OrthoLayout()
    ol.separation(20.0)
    ol.cOverhang(0.4)
    pl.setPlanarLayouter(ol)
    ol.__python_owns__ = False
    pl.call(GA)

In [32]:
# INIT DRAWING BASED ON GRAPHIC FEATURES
def getGraphAttributes(G, isDirectedBenchmark):
    GA = ogdf.GraphAttributes(G, ogdf.GraphAttributes.all)
    for n in G.nodes:
        GA.label[n] = "%s" % n.index()
        GA.fillColor[n] = FILL_COLOR
        GA.height[n] = WIDTH
        GA.width[n] = HEIGHT
        GA.shape[n] = SHAPE

    if not isDirectedBenchmark:
        for e in G.edges:
            GA.arrowType[e] = ogdf.EdgeArrow.__dict__['None']
    
    return GA

In [33]:
# WRITE DRAWING

def writeGraphAttributes(GA, filename, path):
    ogdf.GraphIO.drawSVG( GA, filename+".svg")
    cairosvg.svg2png(url=filename+".svg", write_to=path+"/"+filename+".png", dpi=DPI, background_color=BG, output_width=OUTPUT_WIDTH)

In [34]:
# READ ALL GRAPHS

def readGraphs(graphspath):
    files = [f for f in os.listdir(graphspath) if os.path.isfile(os.path.join(graphspath, f))]
    graphs = []
    for f in files:
        parts = f.split(".")
        ext = parts[len(parts)-1]
        G = ogdf.Graph()
        setattr(G,"label",f.replace(".","_"))
        graphs.append(G)
        if ext.lower() == "lst":
            readFromAdjacencyList(graphspath+"/"+f, G)
        elif ext.lower() == "gml":
            readFromGML(graphspath+"/"+f, G)
    return graphs

In [35]:
# MAIN PROGRAM
def main(graphspath, drawingspath, isDirectedBenchmark):
    graphs = readGraphs(graphspath)
    for G in graphs:
        GA = getGraphAttributes(G, isDirectedBenchmark)
        computeFMMMLayout(GA)
        writeGraphAttributes(GA, f"{G.label}-FMMM", drawingspath)
        if isDirectedBenchmark:
            computeSugiyamaLayout(GA)
            writeGraphAttributes(GA, f"{G.label}-SL", drawingspath)
        else:
            computeOrthoLayout(GA)
            writeGraphAttributes(GA, f"{G.label}-ORTHO", drawingspath)
        

In [ ]:
GRAPHS_PATH = "./graphs"
DRAWINGS_PATH = "./drawings"
IS_DIRECTED_BENCHMARK = False

main(GRAPHS_PATH, DRAWINGS_PATH, IS_DIRECTED_BENCHMARK)